In [13]:
# Since the Skills Network Lab was not loaded, I did this assignment by totally using Jupyter Notebook.
# Thus, appearance of products of each task is supposedly different from those by using the Skills Network because
# I had to create and run dash app in each task to show them, which also updated the appearance of products in previous tasks. 

In [1]:
!pip install pandas dash

In [2]:
!pip install requests

In [3]:
import requests

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv'
response = requests.get(url)

with open('spacex_launch_dash.csv', 'wb') as file:
    file.write(response.content)

print("Download completed!")

Download completed!


In [4]:
import requests

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/spacex_dash_app.py'
response = requests.get(url)

with open('spacex_dash_app.py', 'wb') as file:
    file.write(response.content)

print("Download completed!")

Download completed!


In [5]:
with open('spacex_dash_app.py', 'r') as file:
    code = file.read()

modified_code = code.replace('import dash_html_components as html', 'from dash import html')
modified_code = modified_code.replace('import dash_core_components as dcc', 'from dash import dcc')

exec(modified_code)

In [6]:
# Task 1: Add a Launch Site Dropdown Component

In [7]:
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import pandas as pd

spacex_df = pd.read_csv('spacex_launch_dash.csv')

launch_sites = spacex_df['Launch Site'].unique()

options = [{'label': 'All Sites', 'value': 'ALL'}]
options += [{'label': site, 'value': site} for site in launch_sites]

app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='site-dropdown',
        options=options,
        value='ALL',
        placeholder="Select a Launch Site here",
        searchable=True
    ),
    html.Div(id='dropdown-output-container')
])

@app.callback(
    Output('dropdown-output-container', 'children'),
    Input('site-dropdown', 'value')
)
def update_output(value):
    return f'You have selected "{value}"'

print(dcc.Dropdown(
    id='site-dropdown',
    options=options,
    value='ALL',
    placeholder="Select a Launch Site here",
    searchable=True
))

if __name__ == '__main__':
    app.run_server(debug=True)

Dropdown(options=[{'label': 'All Sites', 'value': 'ALL'}, {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'}, {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'}, {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'}, {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}], value='ALL', searchable=True, placeholder='Select a Launch Site here', id='site-dropdown')


In [8]:
# TASK 2: Add a callback function to render success-pie-chart based on selected site dropdown

In [9]:
import plotly.express as px
import pandas as pd

spacex_df = pd.read_csv('spacex_launch_dash.csv')

app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites', 'value': 'ALL'}
        ] + [{'label': site, 'value': site} for site in spacex_df['Launch Site'].unique()],
        value='ALL',
        placeholder="Select a Launch Site here",
        searchable=True
    ),
    dcc.Graph(id='success-pie-chart')
])

@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    if entered_site == 'ALL':
        # Calculate the distribution of successful launches by site
        fig = px.pie(spacex_df, 
                     names='Launch Site', 
                     values='class', 
                     title='Total Success Launches by Site',
                     labels={'class': 'Success Count'},
                     hole=0.0)
    else:
        filtered_df = spacex_df[spacex_df['Launch Site'] == entered_site]
        success_counts = filtered_df['class'].value_counts().reset_index()
        success_counts.columns = ['class', 'count']
        success_counts['percentage'] = (success_counts['count'] / success_counts['count'].sum()) * 100
        fig = px.pie(success_counts, 
                     names='class', 
                     values='count', 
                     title=f'Total Success Launches for site {entered_site}',
                     labels={'class': 'Launch Outcome'},
                     hole=0.0,
                     hover_data={'count': True})
    
    return fig

if __name__ == '__main__':
    app.run_server(debug=True)

In [10]:
# TASK 3: Add a Range Slider to Select Payload and Task 4 Add a callback function to render the success-payload-scatter-chart scatter plot

In [11]:
min_payload = spacex_df['Payload Mass (kg)'].min()
max_payload = spacex_df['Payload Mass (kg)'].max()

app = dash.Dash(__name__)

app.layout = html.Div([
    dcc.Dropdown(
        id='site-dropdown',
        options=[
            {'label': 'All Sites', 'value': 'ALL'}
        ] + [{'label': site, 'value': site} for site in spacex_df['Launch Site'].unique()],
        value='ALL',
        placeholder="Select a Launch Site here",
        searchable=True
    ),
    dcc.Graph(id='success-pie-chart'),
    dcc.RangeSlider(
        id='payload-slider',
        min=0,
        max=10000,
        step=1000,
        marks={i: f'{i}' for i in range(0, 10001, 2500)},
        value=[min_payload, max_payload]
    ),
    dcc.Graph(id='success-payload-scatter-chart')
])

@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def get_pie_chart(entered_site, payload_range):
    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= payload_range[0]) &
        (spacex_df['Payload Mass (kg)'] <= payload_range[1])
    ]
    
    if entered_site == 'ALL':
        fig = px.pie(filtered_df, 
                     names='Launch Site', 
                     values='class', 
                     title='Total Success Launches by Site',
                     labels={'class': 'Success Count'})
    else:
        filtered_df = filtered_df[filtered_df['Launch Site'] == entered_site]
        success_counts = filtered_df['class'].value_counts().reset_index()
        success_counts.columns = ['class', 'count']
        fig = px.pie(success_counts, 
                     names='class', 
                     values='count', 
                     title=f'Total Success Launches for site {entered_site}',
                     labels={'class': 'Launch Outcome'})
    
    return fig

@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def update_scatter_chart(entered_site, payload_range):
    filtered_df = spacex_df[
        (spacex_df['Payload Mass (kg)'] >= payload_range[0]) &
        (spacex_df['Payload Mass (kg)'] <= payload_range[1])
    ]
    
    if entered_site != 'ALL':
        filtered_df = filtered_df[filtered_df['Launch Site'] == entered_site]
    
    fig = px.scatter(filtered_df, x='Payload Mass (kg)', y='class', 
                     color='Booster Version Category',
                     title='Correlation between Payload and Success for all Sites',
                     labels={'class': 'Launch Outcome'})
    
    return fig

if __name__ == '__main__':
    app.run_server(debug=True)

In [12]:
# Finding Insights Visually

# Now with the dashboard completed, you should be able to use it to analyze SpaceX launch data, and answer the following questions:

   # Which site has the largest successful launches?  --> Selecting All Sites in the drop down menu to show the pie chart gets the answer of KSC LC-39A.

   # Which site has the highest launch success rate? --> Selecting each Site from the dropdown menu and then pointing the shown pie chart to find
   # the counts of success and failure of the site gets the following: 
   # {[Success Launch Counts per Site of KSC LC-39A, CCAFS SLC-40, VAFB SLC-4E, CCAFS LC-40 (10, 3, 4, 7)]/Total Launch Counts (Success + Failure) per Site (13, 7, 10, 26)} X 100(%) = (76.9%, 42.9%, 40%, 26.9%)
   # --> The answer is KSC LC-39A.

   # Which payload range(s) has the highest launch success rate? --> Selecting All Sites in the dropdwon menu, sliding the range on the slider, 
   # and then pointing the shown pie chart gets the following: 
   # {[Success Launch Counts per Range of 0-2500, 2500-5000, 5000-7500, and 7500-10000 in kg (7, 12, 2, 3)]/Total Success Counts (13+7+10+26 = 56)} X 100(%) = (12.5%, 21.43%, 3.57%, 5.36%) 
   # --> The answer is the Range from 2500kg to 5000kg
   # (The range should be 0-2499kg, for example, for mathematical accuracy, but in terms of space payload and for this test, I believe that 
   # the current description does not make a big difference.) 
   # In addition, in the range of 5000 to 7500kg, only KSC LC-39A has the success launches (counts=2), and for the range 7500 to 10000kg, only the VAFB SLC-4E has the success launchs (counts = 3).

   # Which payload range(s) has the lowest launch success rate? --> Similarly, the answer is the Range from 5000kg to 7500kg.

   # Which F9 Booster version (v1.0, v1.1, FT, B4, B5, etc.) has the highest launch success rate? --> Clicking the booster version in the legend,
   # and then counting success versus failure gets the following:
   # {[Success Launch Counts per Booster Version of v1.0, v1.1, FT, B4, and B5 (0, 1, 15, 6, 1)]/Total Launch Counts per Booster (4, 15, 23, 11, 1)} X 100(%) = (0%, 6.67%, 65.2%, 54.5%, 100%) 
   # --> The answer is the booster version B5.
   # However, the total launch counts for this question was 54, not 56, missing two launche counts. 
   # This occurs because the two values of payload mass are the same, the curremt table cannot show the two launch separately
   # as in the Failure of v1.0 and the Success in FT.